In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(5)

SEQ_LEN, STRIDE = 40, 3
CORPUS_CHARS = 200_000
EPOCHS, BATCH = 5, 128

## 1. Load and encode the corpus

In [2]:
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")

with open(path, encoding="utf-8") as f:
    corpus = f.read().lower()[:CORPUS_CHARS]

vocab = sorted(set(corpus))
char_to_id = {ch: i for i, ch in enumerate(vocab)}
id_to_char = np.array(vocab)
print(f"{len(corpus):,} characters, vocabulary size {len(vocab)}")

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
200,000 characters, vocabulary size 37


## 2. Training windows

In [3]:
ids = np.array([char_to_id[ch] for ch in corpus], dtype="int32")
starts = np.arange(0, len(ids) - SEQ_LEN, STRIDE)
X = np.stack([ids[s:s + SEQ_LEN] for s in starts])
y = ids[starts + SEQ_LEN]
print("inputs:", X.shape, "targets:", y.shape)

inputs: (66654, 40) targets: (66654,)


## 3. Model

In [4]:
lstm_model = models.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(len(vocab), 64),
    layers.LSTM(128),
    layers.Dense(len(vocab), activation="softmax"),
])
lstm_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
lstm_model.summary()
lstm_model.fit(X, y, batch_size=BATCH, epochs=EPOCHS, validation_split=0.05)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 37)             │         4,773 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,957 (413.89 KB)

 Trainable params: 105,957 (413.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
495/495 ━━━━━━━━━━━━━━━━━━━━ 45s 87ms/step - loss: 2.6216 - val_loss: 2.3335
Epoch 2/5
495/495 ━━━━━━━━━━━━━━━━━━━━ 82s 87ms/step - loss: 2.2064 - val_loss: 2.1713
Epoch 3/5
495/495 ━━━━━━━━━━━━━━━━━━━━ 43s 87ms/step - loss: 2.0506 - val_loss: 2.0720
Epoch 4/5
495/495 ━━━━━━━━━━━━━━━━━━━━ 44s 89ms/step - loss: 1.9458 - val_loss: 2.0080
Epoch 5/5
495/495 ━━━━━━━━━━━━━━━━━━━━ 44s 88ms/step - loss: 1.8655 - val_loss: 1.9557


## 4. Generation with temperature
Temperature rescales the predicted distribution: low values give safe, repetitive text;
high values give more surprising (and more error-prone) text.

In [5]:
def sample_next(probs, temperature):
    logits = np.log(np.clip(probs, 1e-9, 1.0)) / temperature
    p = np.exp(logits - logits.max())
    p /= p.sum()
    return np.random.choice(len(p), p=p)

def generate(seed, n_chars=300, temperature=0.8):
    seed = seed.lower()
    window = [char_to_id[c] for c in seed if c in char_to_id] or [0]
    out = []
    for _ in range(n_chars):
        ctx = window[-SEQ_LEN:]
        ctx = [0] * (SEQ_LEN - len(ctx)) + ctx             # left-pad short contexts
        probs = lstm_model.predict(np.array([ctx]), verbose=0)[0]
        nxt = sample_next(probs, temperature)
        window.append(nxt)
        out.append(id_to_char[nxt])
    return seed + "".join(out)

## 5. Compare temperatures

In [6]:
seed = "shall i compare thee to a summer's day?\n"
for temp in (0.3, 0.7, 1.0):
    print(f"===== temperature {temp} =====")
    print(generate(seed, n_chars=250, temperature=temp))
    print()

===== temperature 0.3 =====
shall i compare thee to a summer's day?

sicinius:
the comen the carseall and to the minger and the prome the candent to his mare the marcius,
the his the prome the comen the late the comen with the searest the with the comen with the sear the marse the marting the sear the senent the mant

===== temperature 0.7 =====
shall i compare thee to a summer's day?

coriolacus:
till ruster in fire i lore
padsing with
sind easy than chave wothing tithen whis thoust for the more of the gitizensw theath, serecius;
you stand it conest hers sithed,
bet mone and wore
the seat!

clouces:
the arecher,
i consterts the n

===== temperature 1.0 =====
shall i compare thee to a summer's day?
not nemmll tull to thich i the gasnogh begorally; wfou am enour i fithithe say auftour, ins cicheich siten: for stime betour antralanor:
i pay at airploour! nosh, peond this chestar to paplius,
tis hind
oflacet!

 menenantarm:
misees; whour, and mane

